# Durak Deep Monte-Carlo Training Notebook

This notebook mirrors the `train_dmc.py` entry point and lets you launch and monitor Deep Monte-Carlo self-play training directly from a notebook session (locally or on an accelerator such as an H100). The new pipeline relies on ε-greedy self-play, a replay buffer, and Monte-Carlo win/loss returns instead of AlphaZero-style MCTS.

## Environment setup
Run the next cell to move into the repository root so that relative imports work regardless of where the notebook is executed.

In [ ]:
%cd ..


## Imports and utility helpers
The helper below also seeds NumPy and PyTorch for reproducibility across reruns.

In [ ]:
import os
import time
from types import SimpleNamespace

import numpy as np
import torch
import yaml

from agents.qnet import QNet
from rl.trainer import DMCTrainer
from durak.durak_game import initial_state
from durak.encoding import encode_state
from evals.eval_vs_greedy import eval_vs_greedy


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Load configuration
Override any keys in `cfg_overrides` to adapt the run for your hardware (for example, switch to CPU or adjust the batch size).

In [ ]:
with open('configs/durak_dmc.yaml') as f:
    base_cfg = yaml.safe_load(f)

# Edit values in this dictionary to override defaults from the YAML file.
cfg_overrides = {
    # 'device': 'cuda',
    # 'games_per_iter': 500,
    # 'train_steps_per_iter': 1000,
}

cfg_dict = {**base_cfg, **cfg_overrides}
device_str = cfg_dict.get('device', 'cuda')
if device_str != 'cpu' and not torch.cuda.is_available():
    print('CUDA unavailable, falling back to CPU.')
    device_str = 'cpu'
device = torch.device(device_str)
cfg_dict['device'] = device_str
set_seed(cfg_dict['seed'])
cfg = SimpleNamespace(**cfg_dict)

# Create a fresh RNG for both probing the encoder and self-play.
rng = np.random.default_rng(cfg.seed)

## Instantiate the Q-network and trainer
The encoder probe determines the observation size once so that the model matches the environment layout.

In [ ]:
sample_state = initial_state(rng)
input_dim = encode_state(sample_state, perspective_player=0, truesight=True).shape[0]
print(f'Encoded observation dimension: {input_dim}')

qnet = QNet(input_dim, 38).to(device)
trainer = DMCTrainer(qnet, device, cfg)

# Recreate the RNG used for self-play so the initial probe does not change game order.
rng = np.random.default_rng(cfg.seed)

train_state = {
    'total_games': 0,
    'eps': cfg.eps_start,
    'history': [],
}

## Training helpers
`run_training_iterations` performs repeated self-play collection and gradient updates while tracking metrics for plotting.

In [ ]:
def _update_epsilon(total_games: int) -> float:
    if total_games < cfg.eps_decay_games:
        frac = total_games / cfg.eps_decay_games
        return cfg.eps_start + (cfg.eps_end - cfg.eps_start) * frac
    return cfg.eps_end


def run_training_iterations(num_iterations: int = 1, evaluate: bool = False):
    """Run several self-play/optimization iterations and optionally evaluate."""
    metrics = []
    for _ in range(num_iterations):
        truesight = train_state['total_games'] < cfg.truesight_games
        wins, episodes = trainer.selfplay_and_fill(cfg.games_per_iter, train_state['eps'], truesight, rng)
        train_state['total_games'] += episodes

        losses = []
        for _ in range(cfg.train_steps_per_iter):
            loss = trainer.train_step()
            if loss is not None:
                losses.append(loss)

        train_state['eps'] = _update_epsilon(train_state['total_games'])
        iteration_idx = len(train_state['history']) + 1
        record = {
            'iteration': iteration_idx,
            'total_games': train_state['total_games'],
            'epsilon': train_state['eps'],
            'replay_size': len(trainer.replay),
            'selfplay_winrate': wins / episodes if episodes else 0.0,
            'loss_mean': float(np.mean(losses)) if losses else None,
        }
        train_state['history'].append(record)
        metrics.append(record)

        loss_str = f"{record['loss_mean']:.4f}" if record['loss_mean'] is not None else 'n/a'
        msg = (
            f"Iter {iteration_idx:04d} | games={record['total_games']} | eps={record['epsilon']:.3f} | "
            f"replay={record['replay_size']} | selfplay_winrate={record['selfplay_winrate']:.3f} | "
            f"loss={loss_str}"
        )
        print(msg)

        if evaluate:
            wr = eval_vs_greedy(qnet, device, cfg.eval_games)
            print(f"  eval_vs_greedy over {cfg.eval_games} games: {wr:.3f}")

    return metrics

## Launch training
Adjust `num_iterations` as needed. The evaluation call is optional and can be expensive.

In [ ]:
# Example: run a handful of iterations to prime the replay buffer.
run_training_iterations(num_iterations=1, evaluate=False)

## Plot tracked metrics
Feel free to rerun this cell at any time to refresh plots.

In [ ]:
import matplotlib.pyplot as plt

if not train_state['history']:
    print('No training history yet.')
else:
    iterations = [entry['iteration'] for entry in train_state['history']]
    losses = [entry['loss_mean'] for entry in train_state['history']]
    winrates = [entry['selfplay_winrate'] for entry in train_state['history']]
    epsilons = [entry['epsilon'] for entry in train_state['history']]

    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

    axes[0].plot(iterations, losses, marker='o')
    axes[0].set_ylabel('Mean loss')
    axes[0].grid(True)

    axes[1].plot(iterations, winrates, marker='o', color='tab:green')
    axes[1].set_ylabel('Self-play win rate')
    axes[1].set_ylim(0, 1)
    axes[1].grid(True)

    axes[2].plot(iterations, epsilons, marker='o', color='tab:red')
    axes[2].set_ylabel('ε (exploration)')
    axes[2].set_xlabel('Iteration')
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()

## On-demand evaluation vs. greedy baseline
Run this cell to benchmark the current network without exploration.

In [ ]:
wr = eval_vs_greedy(qnet, device, cfg.eval_games)
print(f'Greedy baseline win rate over {cfg.eval_games} games: {wr:.3f}')

## Save or resume checkpoints
The checkpoint files are fully compatible with `train_dmc.py` and the CLI workflow.

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
checkpoint_path = 'checkpoints/qnet_notebook.pt'
torch.save(qnet.state_dict(), checkpoint_path)
print(f'Saved checkpoint to {checkpoint_path}')

In [ ]:
# To resume from a saved state, uncomment and set the path:
# checkpoint_path = 'checkpoints/qnet_iter10.pt'
# qnet.load_state_dict(torch.load(checkpoint_path, map_location=device))
# qnet.to(device)
# qnet.eval()